# Atlas ensemble baselines — 75 stochastic members per event

For every atlas event, unrolls **75 realizations of the unperturbed initial condition** through NeuralGCM's stochastic physics — the internal-variability envelope the optimized storyline must beat. This is the like-for-like version of W&DL's +3.7 °C benchmark (optimized peak vs the hottest ensemble member), and the per-zone spread makes storyline gains comparable across climate zones (`gain_over_spread`).

**Prerequisite:** the optimization runs from `run_atlas.ipynb` (each event needs its `storyline.csv`, restored from Drive).

**Before running:** Runtime → Change runtime type → **GPU (A100 or L4, High-RAM)**.

**Cost & resume:** members are forward-only (no gradients) — a 75-member ensemble is roughly a quarter to a third of the optimization already spent per event. Each member's trajectory streams to Drive **as it completes** (`heatwave_atlas/ensembles/EXP75/<event>/member_NNN.csv`), so a killed session resumes at the exact member it died on — just re-run this notebook until every event is done.

In [1]:
REPO_URL = "https://github.com/ieadoboe/heatwave-initial-conditions.git"

import shutil, sys
from pathlib import Path

name = Path(REPO_URL).stem
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / name]
root = next((p for p in candidates if (p / "heatwave_ic").is_dir()), None)
if root is None and "google.colab" in sys.modules:
    shutil.rmtree(name, ignore_errors=True)   # clear stale/partial clones
    !git clone {REPO_URL}
    root = Path.cwd() / name
if root is None or not (root / "heatwave_ic").is_dir():
    raise RuntimeError("heatwave_ic/ not found — clone failed or package not pushed.")
%cd {root}
sys.path.insert(0, str(root))
if "google.colab" in sys.modules:
    %pip install -q -U neuralgcm dinosaur gcsfs optax tqdm pyyaml zarr

Cloning into 'heatwave-initial-conditions'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 270 (delta 65), reused 69 (delta 39), pack-reused 171 (from 1)
Receiving objects: 100% (270/270), 169.40 MiB | 18.42 MiB/s, done.
Resolving deltas: 100% (147/147), done.
/content/heatwave-initial-conditions
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 57.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Drive persistence — must point at the SAME folder run_atlas used, so the
# completed optimization runs (storyline.csv) are restored here.
PERSIST = True
PERSIST_DIR = "/content/drive/MyDrive/heatwave_atlas"

if PERSIST and "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(PERSIST_DIR).mkdir(parents=True, exist_ok=True)
persist_flag = f"--persist-dir {PERSIST_DIR}" if PERSIST else ""

# The merge step needs the atlas summary present locally too.
if PERSIST and Path(PERSIST_DIR, "atlas_summary.csv").exists():
    Path("data").mkdir(exist_ok=True)
    shutil.copy2(Path(PERSIST_DIR, "atlas_summary.csv"), "data/atlas_summary.csv")

Mounted at /content/drive


## Run the ensembles

Budget ~1–2 events per session, same as the atlas — use the subset form to pick this session's events.

In [3]:
!python scripts/run_ensembles.py {persist_flag}

# Subset form — e.g. just the PNW validation ensemble:
# !python scripts/run_ensembles.py {persist_flag} --configs configs/pnw_jun2021.yaml

# Recompute an event's ensemble from scratch:
# !python scripts/run_ensembles.py {persist_flag} --rerun --configs configs/pnw_jun2021.yaml

Restoring runs from /content/drive/MyDrive/heatwave_atlas ...

pnw_jun2021: target (50.231, -121.581°E), event 2021-06-25 → 2021-07-01 (peak 2021-06-29)
  init 2021-06-20 (lead 9 d), evolve 11 d, target window = last 5 d
  beta=10 lambda=20 T_ref=298.15 lr=1e-09 iters=75
  model v1_precip/stochastic_precip_2_8_deg.pkl  IC data/era5_ic_pnw_jun2021_2021-06-20.zarr
Loading model v1_precip/stochastic_precip_2_8_deg.pkl ...
Opening ARCO-ERA5 and slicing the IC window at 2021-06-20 ...
Materialising 25 snapshots -> data/era5_ic_pnw_jun2021_2021-06-20.zarr ...
^C


In [4]:
import pandas as pd

summary = pd.read_csv("data/atlas_summary.csv")
cols = ["event", "zone", "storyline_gain_C", "ens_max_peak_C",
        "gain_vs_ens_max_C", "ens_peak_spread_C", "gain_over_spread"]
display(summary[[c for c in cols if c in summary.columns]])

print("\nEnsemble figures written to plots/:")
for p in sorted(Path("plots").glob("*_ensemble.pdf")):
    print(" ", p)

,event,zone,storyline_gain_C
0,pnw_jun2021,maritime coastal,2.91
1,stjohns_aug2025,maritime coastal,1.04
2,moscow_jul2010,continental interior,2.34
3,japan_jul2018,subtropical humid,3.35
4,sahel_apr2024,arid,2.78
5,brazil_nov2023,tropical,1.52
6,siberia_jun2020,polar/high-latitude,6.70



Ensemble figures written to plots/:


## Reading the result

- **`gain_vs_ens_max_C`** — the W&DL quantity: how far the optimized storyline peaks *above the hottest of 75 members*. Positive = the storyline is genuinely beyond internal variability. For `pnw_jun2021`, this is the number to compare with the paper's **+3.7 °C**.
- **`gain_over_spread`** — the cross-zone-comparable headline: storyline gain in units of the zone's own internal variability. This is the column that fairly ranks the climate zones.
- Per-event member trajectories are in each run dir's `ensemble.csv` (75 columns); the `*_ensemble.pdf` figures show the optimized trajectory against the gray member envelope.